In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf


In [ ]:
def load_jpm_dividend_history(
    end_date: pd.Timestamp,
) -> pd.DataFrame:
    ticker = yf.Ticker("JPM")
    dividends = ticker.dividends

    if dividends.empty:
        raise ValueError("No JPM dividend history was returned.")

    dividends = dividends.copy()
    dividends.index = pd.to_datetime(dividends.index)

    if dividends.index.tz is not None:
        dividends.index = dividends.index.tz_localize(None)

    dividend_df = (
        dividends.loc[dividends.index <= end_date]
        .rename("Dividend")
        .reset_index()
    )

    dividend_df.columns = ["Date", "Dividend"]

    dividend_df["Date"] = (
        pd.to_datetime(dividend_df["Date"])
        .dt.tz_localize(None)
        .dt.normalize()
        .astype("datetime64[ns]")
    )

    return dividend_df


In [ ]:
def build_week2_pipeline(
    input_file: str,
    output_file: str,
) -> pd.DataFrame:

    input_path = Path(input_file)
    output_path = Path(output_file)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file was not found: {input_path.resolve()}"
        )

    df = pd.read_csv(input_path)

    df.columns = [
        str(column).strip()
        for column in df.columns
    ]

    if "Adj_Close" in df.columns and "Adj Close" not in df.columns:
        df = df.rename(
            columns={"Adj_Close": "Adj Close"}
        )

    df["Date"] = (
        pd.to_datetime(df["Date"], errors="coerce")
        .dt.tz_localize(None)
        .dt.normalize()
        .astype("datetime64[ns]")
    )

    df = (
        df.dropna(subset=["Date"])
        .sort_values("Date")
        .drop_duplicates(subset=["Date"], keep="last")
        .reset_index(drop=True)
    )

    numeric_columns = [
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        "VIX",
        "Treasury_10Y",
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

    df[numeric_columns] = df[numeric_columns].ffill()

    df["Daily_Return"] = df["Close"].pct_change()
    df["Log_Return"] = np.log(
        df["Close"] / df["Close"].shift(1)
    )

    df["VIX_Change"] = df["VIX"].pct_change()
    df["Rate_Change"] = df["Treasury_10Y"].diff()
    df["Volume_Change"] = df["Volume"].pct_change()

    for window in [5, 20, 60]:
        df[f"Rolling_Vol_{window}D"] = (
            df["Log_Return"]
            .rolling(window=window)
            .std()
            * np.sqrt(252)
        )

    dividend_df = load_jpm_dividend_history(
        end_date=df["Date"].max()
    )

    df["Date"] = df["Date"].astype("datetime64[ns]")
    dividend_df["Date"] = dividend_df["Date"].astype(
        "datetime64[ns]"
    )

    df = pd.merge_asof(
        df.sort_values("Date"),
        dividend_df.sort_values("Date"),
        on="Date",
        direction="backward",
    )

    df["Dividend"] = df["Dividend"].ffill().fillna(0)

    df["Dividend_Growth"] = 0.0
    dividend_changed = df["Dividend"].ne(
        df["Dividend"].shift(1)
    )
    previous_dividend = df["Dividend"].shift(1)

    valid_growth = (
        dividend_changed
        & previous_dividend.gt(0)
    )

    df.loc[valid_growth, "Dividend_Growth"] = (
        df.loc[valid_growth, "Dividend"]
        / previous_dividend.loc[valid_growth]
        - 1
    )

    df["VIX_Return"] = df["VIX"].pct_change()

    df["VIX_JPM_Correlation_20D"] = (
        df["Daily_Return"]
        .rolling(window=20)
        .corr(df["VIX_Return"])
    )

    df["Rate_Momentum_1D"] = (
        df["Treasury_10Y"].diff(1)
    )
    df["Rate_Momentum_5D"] = (
        df["Treasury_10Y"].diff(5)
    )
    df["Rate_Momentum_20D"] = (
        df["Treasury_10Y"].diff(20)
    )

    df["Rate_Pct_Change_5D"] = (
        df["Treasury_10Y"].pct_change(5)
    )
    df["Rate_Pct_Change_20D"] = (
        df["Treasury_10Y"].pct_change(20)
    )

    df = df.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    exact_columns = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        "VIX",
        "Treasury_10Y",
        "Daily_Return",
        "Log_Return",
        "VIX_Change",
        "Rate_Change",
        "Volume_Change",
        "Rolling_Vol_5D",
        "Rolling_Vol_20D",
        "Rolling_Vol_60D",
        "Dividend",
        "Dividend_Growth",
        "VIX_Return",
        "VIX_JPM_Correlation_20D",
        "Rate_Momentum_1D",
        "Rate_Momentum_5D",
        "Rate_Momentum_20D",
        "Rate_Pct_Change_5D",
        "Rate_Pct_Change_20D",
    ]

    df = (
        df.dropna(
            subset=[
                "Rolling_Vol_60D",
                "VIX_JPM_Correlation_20D",
                "Rate_Momentum_20D",
                "Rate_Pct_Change_20D",
            ]
        )
        .loc[:, exact_columns]
        .reset_index(drop=True)
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df.to_csv(
        output_path,
        index=False,
    )

    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print(f"Saved to: {output_path.resolve()}")

    return df


In [ ]:
INPUT_FILE = "data/week1_initial_raw_dataset.csv"
OUTPUT_FILE = "output/week2_feature_dataset_pipeline.csv"

week2_data_pipeline = build_week2_pipeline(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
)

week2_data_pipeline.head()
